# traineo1 Benchmark

Notebook ini adalah duplicate yang disederhanakan dari pola `traineo1 kaggle clean`.

Tetap dipertahankan:
- `.venv` di dalam repo
- `uv` untuk install dan menjalankan command
- hard fail kalau GPU kurang dari 2
- training nyata via `accelerate --num_processes=2`

Yang disesuaikan:
- baseline pakai `attn_backend=torch`
- hybrid pakai `attn_backend=flash_attn`
- hybrid tetap teacher-distilled dari original F5-TTS checkpoint
- kedua model disiapkan untuk `WER`, `SIM-o`, `SMOS`, dan `CMOS`

In [ ]:
# ── Cell 1: Config & Helpers ──────────────────────────────────────────────
import csv
import json
import os
import shutil
import subprocess
from itertools import chain
from pathlib import Path


def optional_str(value: str) -> str:
    return (value or '').strip()


def detect_repo_dir() -> Path:
    cwd = Path.cwd().resolve()
    for cand in [cwd, *cwd.parents]:
        if (cand / 'src' / 'f5_tts').exists() and (cand / 'pyproject.toml').exists():
            return cand
    raise FileNotFoundError('Repo gardenbunga tidak terdeteksi. Jalankan notebook ini dari repo.')


REPO_DIR = detect_repo_dir()
VENV_DIR = REPO_DIR / '.venv'
VENV_PY = VENV_DIR / 'bin/python'

DATASET_NAME = 'datasetku'
DATASET_ROOT = Path('/kaggle/input/tts-indo')
DATASET_SUBDIR = 'data'
CSV_1 = Path('') if False else None
CSV_2 = Path('') if False else None

PRETRAIN_LOCAL_CKPT = None
HF_PRETRAIN_REPO_ID = 'SWivid/F5-TTS'
HF_PRETRAIN_FILENAME = 'F5TTS_v1_Base/model_1250000.safetensors'
HF_TOKEN = ''

EVAL_SPECS = [
    {
        'name': 'seedtts_test_en',
        'task_type': 'seedtts',
        'lang': 'en',
        'meta_file': '/kaggle/input/seed-tts-eval/seedtts_testset/en/meta.lst',
        'librispeech_test_clean_path': '',
    },
]
ASR_CKPT = None
WAVLM_CKPT = Path('/kaggle/input/wavlm-large-finetune/wavlm_large_finetune.pth')
SMOS_RATINGS = None
CMOS_RATINGS = None

ACCELERATE_NUM_PROCESSES = 2
ACCELERATE_MIXED_PRECISION = 'fp16'
TRAIN_NUM_WORKERS = 4
TRAIN_EPOCHS = 11
TRAIN_LR = 7.5e-5
TRAIN_WEIGHT_DECAY = 0.01
TRAIN_WARMUP_UPDATES = 20000
TRAIN_GRAD_ACCUMULATION_STEPS = 1
TRAIN_MAX_GRAD_NORM = 1.0
TRAIN_BATCH_SIZE_PER_GPU = 4096
TRAIN_MAX_SAMPLES = 32
TRAIN_SAVE_PER_UPDATES = 10000
TRAIN_LAST_PER_UPDATES = 1000
TRAIN_KEEP_LAST = 2

BENCHMARK_BATCH_SIZE = 1
BENCHMARK_FRAME_LENGTH = 256
BENCHMARK_LONG_FRAME_LENGTH = 1024
BENCHMARK_TEXT_LENGTH = 80
BENCHMARK_SAMPLE_STEPS = 4
BENCHMARK_WARMUP_ITERS = 1
BENCHMARK_ITERS = 3

INFER_SEED = 0
INFER_NFE_STEP = 16
EVAL_GPU_LIST = '[0,1]'
RECREATE_VENV = False

DATASET_DATA_DIR = DATASET_ROOT / DATASET_SUBDIR
if CSV_1 is None:
    auto_csv = sorted(DATASET_ROOT.glob('**/metadata.csv'))
    CSV_1 = auto_csv[0] if auto_csv else None

NOTEBOOK_TAG = 'traineo1_benchmark'
CONFIG_DIR = REPO_DIR / 'src' / 'f5_tts' / 'configs'
BASELINE_CONFIG_NAME = 'traineo1_benchmark_baseline.yaml'
HYBRID_CONFIG_NAME = 'traineo1_benchmark_hybrid.yaml'
BASELINE_CONFIG_PATH = CONFIG_DIR / BASELINE_CONFIG_NAME
HYBRID_CONFIG_PATH = CONFIG_DIR / HYBRID_CONFIG_NAME

BASELINE_SAVE_DIR = REPO_DIR / 'ckpts' / NOTEBOOK_TAG / 'baseline'
HYBRID_SAVE_DIR = REPO_DIR / 'ckpts' / NOTEBOOK_TAG / 'hybrid'
PRETRAIN_CACHE_DIR = REPO_DIR / 'ckpts' / NOTEBOOK_TAG / 'pretrained'
PRETRAIN_TARGET_CKPT = PRETRAIN_CACHE_DIR / 'model_1250000.safetensors'
PREPARED_DATASET_DIR = REPO_DIR / 'data' / f'{DATASET_NAME}_pinyin'
MERGED_CSV = REPO_DIR / 'data' / f'{DATASET_NAME}_merged_metadata.csv'
RESULTS_ROOT = REPO_DIR / 'results' / NOTEBOOK_TAG
GENERATED_ROOT = RESULTS_ROOT / 'generated'
SUBJECTIVE_ROOT = RESULTS_ROOT / 'subjective'
MATRIX_ROOT = RESULTS_ROOT / 'matrix'
BENCHMARK_JSON = RESULTS_ROOT / 'benchmark.json'

for path in [BASELINE_SAVE_DIR, HYBRID_SAVE_DIR, PRETRAIN_CACHE_DIR, RESULTS_ROOT, GENERATED_ROOT, SUBJECTIVE_ROOT, MATRIX_ROOT]:
    path.mkdir(parents=True, exist_ok=True)


def build_runtime_env() -> dict:
    env = os.environ.copy()
    site_packages = sorted((VENV_DIR / 'lib').glob('python*/site-packages'))
    if site_packages:
        sp = site_packages[-1]
        cuda_rel = [
            'nvidia/cublas/lib',
            'nvidia/cuda_runtime/lib',
            'nvidia/cudnn/lib',
            'nvidia/cufft/lib',
            'nvidia/nccl/lib',
            'nvidia/nvjitlink/lib',
        ]
        cuda_libs = [str(sp / rel) for rel in cuda_rel if (sp / rel).exists()]
        if cuda_libs:
            current = env.get('LD_LIBRARY_PATH', '')
            env['LD_LIBRARY_PATH'] = ':'.join(cuda_libs + ([current] if current else []))
    env['PYTHONNOUSERSITE'] = '1'
    env['PYTHONPATH'] = str(REPO_DIR / 'src')
    env['CUDA_VISIBLE_DEVICES'] = '0,1'
    env['TOKENIZERS_PARALLELISM'] = 'false'
    env['PYTHONFAULTHANDLER'] = '1'
    env['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True,roundup_power2_divisions:16'
    env['TORCH_ALLOW_TF32_CUBLAS_OVERRIDE'] = '1'
    env['PIP_DISABLE_PIP_VERSION_CHECK'] = '1'
    env['HF_HUB_DISABLE_TELEMETRY'] = '1'
    if HF_TOKEN:
        env['HF_TOKEN'] = HF_TOKEN
        env['HUGGING_FACE_HUB_TOKEN'] = HF_TOKEN
    return env


def run_cmd(cmd, cwd=None, env=None, capture_output=False):
    printable = cmd if isinstance(cmd, str) else ' '.join(str(x) for x in cmd)
    print('$', printable)
    return subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        env=env,
        check=True,
        text=True,
        capture_output=capture_output,
    )


def run_py(args, cwd=None, env=None, capture_output=False):
    return run_cmd(['uv', 'run', '--python', str(VENV_PY), *args], cwd=cwd, env=env or build_runtime_env(), capture_output=capture_output)


def run_py_nosync(args, cwd=None, env=None, capture_output=False):
    return run_cmd(['uv', 'run', '--no-sync', '--python', str(VENV_PY), *args], cwd=cwd, env=env or build_runtime_env(), capture_output=capture_output)


def resolve_latest_checkpoint(save_dir: Path) -> Path:
    last = save_dir / 'model_last.pt'
    if last.exists():
        return last
    pts = sorted(save_dir.glob('model_*.pt'))
    if pts:
        return pts[-1]
    raise FileNotFoundError(f'Checkpoint tidak ditemukan di {save_dir}')


def load_rows(csv_path: Path):
    rows = []
    with open(csv_path, 'r', encoding='utf-8-sig', newline='') as handle:
        reader = csv.reader(handle, delimiter='|')
        first = next(reader, None)
        if first is None:
            return rows
        has_header = len(first) >= 2 and first[0].strip() == 'audio_file' and first[1].strip() == 'text'
        iterator = reader if has_header else chain([first], reader)
        for row in iterator:
            if len(row) >= 2 and row[0].strip() and row[1].strip():
                rows.append({'audio_file': row[0].strip(), 'text': row[1].strip()})
    return rows


def resolve_audio_path(raw_path: str, csv_path: Path) -> Path:
    candidate = Path(raw_path).expanduser()
    if candidate.is_absolute() and candidate.exists():
        return candidate.resolve()
    for base in [csv_path.parent, DATASET_DATA_DIR, DATASET_ROOT, REPO_DIR]:
        full = (base / raw_path).expanduser()
        if full.exists():
            return full.resolve()
    return (DATASET_ROOT / raw_path).expanduser().resolve()


print('REPO_DIR     :', REPO_DIR)
print('VENV_DIR     :', VENV_DIR)
print('DATASET_ROOT :', DATASET_ROOT)
print('RESULTS_ROOT :', RESULTS_ROOT)


In [ ]:
# ── Cell 2: Verifikasi Repo dan Dataset ───────────────────────────────────
required_repo = [
    REPO_DIR / 'pyproject.toml',
    REPO_DIR / 'src/f5_tts/train/train.py',
    REPO_DIR / 'src/f5_tts/model/cfm.py',
    REPO_DIR / 'src/f5_tts/model/trainer.py',
    REPO_DIR / 'src/f5_tts/scripts/benchmark_hybrid_mamba.py',
    REPO_DIR / 'src/f5_tts/scripts/prepare_subjective_eval.py',
    REPO_DIR / 'src/f5_tts/scripts/build_eval_matrix.py',
    REPO_DIR / 'src/f5_tts/configs/F5TTS_v1_Base.yaml',
    REPO_DIR / 'src/f5_tts/configs/F5TTS_v1_Base_Mamba_Conservative.yaml',
    REPO_DIR / 'data/Emilia_ZH_EN_pinyin/vocab.txt',
]
missing_repo = [str(path) for path in required_repo if not path.exists()]
if missing_repo:
    raise FileNotFoundError('Repo asset belum lengkap:\n' + '\n'.join(missing_repo))

if not DATASET_ROOT.exists():
    raise FileNotFoundError(f'Dataset root tidak ditemukan: {DATASET_ROOT}')
if CSV_1 is None or not CSV_1.exists():
    raise FileNotFoundError('metadata.csv tidak ditemukan. Isi path CSV_1 secara eksplisit.')
if CSV_2 is not None and not CSV_2.exists():
    raise FileNotFoundError(f'CSV_2 tidak ditemukan: {CSV_2}')

for spec in EVAL_SPECS:
    if not Path(spec['meta_file']).exists():
        raise FileNotFoundError(f"meta_file tidak ditemukan: {spec['meta_file']}")
    if spec['task_type'] == 'librispeech' and not Path(spec['librispeech_test_clean_path']).exists():
        raise FileNotFoundError(f"librispeech_test_clean_path tidak ditemukan: {spec['librispeech_test_clean_path']}")
if WAVLM_CKPT is None or not WAVLM_CKPT.exists():
    raise FileNotFoundError('Isi WAVLM_CKPT dengan checkpoint WavLM lokal.')

run_cmd(['git', '-C', str(REPO_DIR), 'rev-parse', '--short', 'HEAD'])
print('CSV_1:', CSV_1)
print('CSV_2:', CSV_2 if CSV_2 else '(none)')


In [ ]:
# ── Cell 3: Setup venv + dependencies ─────────────────────────────────────
if shutil.which('uv') is None:
    run_cmd(['python3', '-m', 'pip', 'install', '-U', 'uv'])

if shutil.which('apt-get') is not None:
    run_cmd(['apt-get', 'update', '-y'])
    run_cmd(['apt-get', 'install', '-y', 'python3.11', 'python3.11-venv', 'python3.11-dev', 'build-essential'])

if RECREATE_VENV and VENV_DIR.exists():
    shutil.rmtree(VENV_DIR)

if not VENV_PY.exists():
    run_cmd(['uv', 'venv', '--python', '/usr/bin/python3.11', str(VENV_DIR)])
else:
    print('Reuse existing venv:', VENV_DIR)

run_cmd(['uv', 'pip', 'install', '--python', str(VENV_PY), '--upgrade', 'pip', 'wheel', 'setuptools<82'])
run_cmd([
    'uv', 'pip', 'install', '--python', str(VENV_PY),
    '--index-url', 'https://download.pytorch.org/whl/cu128',
    'torch==2.8.0+cu128', 'torchvision==0.23.0+cu128', 'torchaudio==2.8.0+cu128'
])
run_cmd(['uv', 'pip', 'install', '--python', str(VENV_PY), '-e', '.[eval]'], cwd=REPO_DIR)
run_cmd(['uv', 'pip', 'install', '--python', str(VENV_PY), 'ninja', 'huggingface_hub', 'ctranslate2==4.5.0'], cwd=REPO_DIR)

install_env = build_runtime_env()
install_env['PIP_NO_DEPS'] = '1'
causal_whl = 'https://github.com/Dao-AILab/causal-conv1d/releases/download/v1.6.1.post4/causal_conv1d-1.6.1+cu12torch2.8cxx11abiTRUE-cp311-cp311-linux_x86_64.whl'
mamba_whl = 'https://github.com/state-spaces/mamba/releases/download/v2.3.1/mamba_ssm-2.3.1+cu12torch2.8cxx11abiTRUE-cp311-cp311-linux_x86_64.whl'
run_cmd(['uv', 'run', '--no-sync', '--python', str(VENV_PY), 'pip', 'uninstall', '-y', 'mamba-ssm', 'causal-conv1d'], cwd=REPO_DIR, env=install_env)
run_cmd(['uv', 'run', '--no-sync', '--python', str(VENV_PY), 'pip', 'install', '--no-deps', '--force-reinstall', causal_whl], cwd=REPO_DIR, env=install_env)
run_cmd(['uv', 'run', '--no-sync', '--python', str(VENV_PY), 'pip', 'install', '--no-deps', '--force-reinstall', mamba_whl], cwd=REPO_DIR, env=install_env)
run_py_nosync(['-c', "import torch, accelerate, mamba_ssm; print(torch.__version__); print(accelerate.__version__); print(mamba_ssm.__version__)"], cwd=REPO_DIR)


In [ ]:
# ── Cell 4: Hard fail kalau bukan 2 GPU ───────────────────────────────────
gpu_check = run_py(['-c', "import torch; print('gpu_count =', torch.cuda.device_count()); [print('gpu', i, torch.cuda.get_device_name(i)) for i in range(torch.cuda.device_count())]"], cwd=REPO_DIR, capture_output=True)
print(gpu_check.stdout)
lines = [line.strip() for line in gpu_check.stdout.splitlines() if line.strip()]
count_line = [line for line in lines if line.startswith('gpu_count =')]
if not count_line:
    raise RuntimeError('Gagal membaca jumlah GPU.')
gpu_count = int(count_line[0].split('=')[1].strip())
if gpu_count < 2:
    raise RuntimeError(f'Notebook ini wajib 2 GPU tanpa fallback. Terdeteksi cuma {gpu_count}.')
if ACCELERATE_NUM_PROCESSES != 2:
    raise RuntimeError('ACCELERATE_NUM_PROCESSES harus 2.')


In [ ]:
# ── Cell 5: Pretrained checkpoint + merged metadata + prepared dataset ────
if PRETRAIN_LOCAL_CKPT is not None:
    if not PRETRAIN_LOCAL_CKPT.exists():
        raise FileNotFoundError(f'Pretrain lokal tidak ditemukan: {PRETRAIN_LOCAL_CKPT}')
    shutil.copy2(PRETRAIN_LOCAL_CKPT, PRETRAIN_TARGET_CKPT)
else:
    from huggingface_hub import hf_hub_download
    downloaded = hf_hub_download(
        repo_id=HF_PRETRAIN_REPO_ID,
        filename=HF_PRETRAIN_FILENAME,
        token=HF_TOKEN or None,
        local_dir=PRETRAIN_CACHE_DIR,
        local_dir_use_symlinks=False,
    )
    downloaded = Path(downloaded)
    if downloaded.resolve() != PRETRAIN_TARGET_CKPT.resolve():
        shutil.copy2(downloaded, PRETRAIN_TARGET_CKPT)

merged_rows = []
seen = set()
for csv_path in [CSV_1, CSV_2]:
    if csv_path is None:
        continue
    for row in load_rows(csv_path):
        audio_path = resolve_audio_path(row['audio_file'], csv_path)
        if not audio_path.exists():
            continue
        audio_key = str(audio_path)
        if audio_key in seen:
            continue
        seen.add(audio_key)
        merged_rows.append({'audio_file': audio_key, 'text': row['text']})

if not merged_rows:
    raise RuntimeError('Metadata gabungan kosong.')

MERGED_CSV.parent.mkdir(parents=True, exist_ok=True)
with open(MERGED_CSV, 'w', encoding='utf-8', newline='') as handle:
    writer = csv.DictWriter(handle, fieldnames=['audio_file', 'text'], delimiter='|')
    writer.writeheader()
    writer.writerows(merged_rows)

run_py_nosync([
    'src/f5_tts/train/datasets/prepare_csv_wavs.py',
    str(MERGED_CSV),
    str(PREPARED_DATASET_DIR),
    '--workers', str(TRAIN_NUM_WORKERS),
], cwd=REPO_DIR)

required_dataset = [PRETRAIN_TARGET_CKPT, PREPARED_DATASET_DIR / 'duration.json', PREPARED_DATASET_DIR / 'raw.arrow', PREPARED_DATASET_DIR / 'vocab.txt']
missing_dataset = [str(path) for path in required_dataset if not path.exists()]
if missing_dataset:
    raise FileNotFoundError('Asset training belum lengkap:\n' + '\n'.join(missing_dataset))

print('PRETRAIN_TARGET_CKPT:', PRETRAIN_TARGET_CKPT)
print('MERGED_CSV          :', MERGED_CSV)
print('PREPARED_DATASET    :', PREPARED_DATASET_DIR)


In [ ]:
# ── Cell 6: Flash-Attention wajib aktif untuk hybrid ──────────────────────
install_env = build_runtime_env()
install_env['PIP_NO_DEPS'] = '1'
flash_whl = 'https://github.com/Dao-AILab/flash-attention/releases/download/v2.8.3/flash_attn-2.8.3+cu12torch2.8cxx11abiTRUE-cp311-cp311-linux_x86_64.whl'
run_cmd(['uv', 'run', '--no-sync', '--python', str(VENV_PY), 'pip', 'uninstall', '-y', 'flash-attn', 'flash_attn'], cwd=REPO_DIR, env=install_env)
run_cmd(['uv', 'run', '--no-sync', '--python', str(VENV_PY), 'pip', 'install', '--no-deps', '--force-reinstall', flash_whl], cwd=REPO_DIR, env=install_env)
run_py_nosync(['-c', "import flash_attn; print('flash_attn', flash_attn.__version__)"], cwd=REPO_DIR)


In [ ]:
# ── Cell 7: Tulis runtime config baseline + hybrid ────────────────────────
from omegaconf import OmegaConf

baseline_cfg = OmegaConf.load(REPO_DIR / 'src/f5_tts/configs/F5TTS_v1_Base.yaml')
baseline_cfg.model.name = 'F5TTS_v1_Base_Kaggle_Benchmark_Baseline'
baseline_cfg.datasets.name = DATASET_NAME
baseline_cfg.datasets.batch_size_per_gpu = TRAIN_BATCH_SIZE_PER_GPU
baseline_cfg.datasets.batch_size_type = 'frame'
baseline_cfg.datasets.max_samples = TRAIN_MAX_SAMPLES
baseline_cfg.datasets.num_workers = TRAIN_NUM_WORKERS
baseline_cfg.optim.epochs = TRAIN_EPOCHS
baseline_cfg.optim.learning_rate = TRAIN_LR
baseline_cfg.optim.weight_decay = TRAIN_WEIGHT_DECAY
baseline_cfg.optim.num_warmup_updates = TRAIN_WARMUP_UPDATES
baseline_cfg.optim.grad_accumulation_steps = TRAIN_GRAD_ACCUMULATION_STEPS
baseline_cfg.optim.max_grad_norm = TRAIN_MAX_GRAD_NORM
baseline_cfg.optim.mixed_precision = ACCELERATE_MIXED_PRECISION
baseline_cfg.model.arch.attn_backend = 'torch'
baseline_cfg.model.arch.checkpoint_activations = True
baseline_cfg.ckpts.logger = None
baseline_cfg.ckpts.log_samples = False
baseline_cfg.ckpts.save_per_updates = TRAIN_SAVE_PER_UPDATES
baseline_cfg.ckpts.last_per_updates = TRAIN_LAST_PER_UPDATES
baseline_cfg.ckpts.keep_last_n_checkpoints = TRAIN_KEEP_LAST
baseline_cfg.ckpts.save_dir = BASELINE_SAVE_DIR.relative_to(REPO_DIR).as_posix()
baseline_cfg.ckpts.student_init_checkpoint = str(PRETRAIN_TARGET_CKPT)
baseline_cfg.ckpts.load_ema_student_init = True
OmegaConf.save(baseline_cfg, BASELINE_CONFIG_PATH)

hybrid_cfg = OmegaConf.load(REPO_DIR / 'src/f5_tts/configs/F5TTS_v1_Base_Mamba_Conservative.yaml')
hybrid_cfg.model.name = 'F5TTS_v1_Base_Kaggle_Benchmark_Hybrid'
hybrid_cfg.datasets.name = DATASET_NAME
hybrid_cfg.datasets.batch_size_per_gpu = TRAIN_BATCH_SIZE_PER_GPU
hybrid_cfg.datasets.batch_size_type = 'frame'
hybrid_cfg.datasets.max_samples = TRAIN_MAX_SAMPLES
hybrid_cfg.datasets.num_workers = TRAIN_NUM_WORKERS
hybrid_cfg.optim.epochs = TRAIN_EPOCHS
hybrid_cfg.optim.learning_rate = TRAIN_LR
hybrid_cfg.optim.weight_decay = TRAIN_WEIGHT_DECAY
hybrid_cfg.optim.num_warmup_updates = TRAIN_WARMUP_UPDATES
hybrid_cfg.optim.grad_accumulation_steps = TRAIN_GRAD_ACCUMULATION_STEPS
hybrid_cfg.optim.max_grad_norm = TRAIN_MAX_GRAD_NORM
hybrid_cfg.optim.mixed_precision = ACCELERATE_MIXED_PRECISION
hybrid_cfg.model.arch.attn_backend = 'flash_attn'
hybrid_cfg.model.arch.checkpoint_activations = True
hybrid_cfg.ckpts.logger = None
hybrid_cfg.ckpts.log_samples = False
hybrid_cfg.ckpts.save_per_updates = TRAIN_SAVE_PER_UPDATES
hybrid_cfg.ckpts.last_per_updates = TRAIN_LAST_PER_UPDATES
hybrid_cfg.ckpts.keep_last_n_checkpoints = TRAIN_KEEP_LAST
hybrid_cfg.ckpts.save_dir = HYBRID_SAVE_DIR.relative_to(REPO_DIR).as_posix()
hybrid_cfg.ckpts.teacher_checkpoint = str(PRETRAIN_TARGET_CKPT)
hybrid_cfg.ckpts.student_init_checkpoint = str(PRETRAIN_TARGET_CKPT)
hybrid_cfg.ckpts.load_ema_teacher = True
hybrid_cfg.ckpts.load_ema_student_init = True
OmegaConf.save(hybrid_cfg, HYBRID_CONFIG_PATH)

print('BASELINE_CONFIG_PATH:', BASELINE_CONFIG_PATH)
print('HYBRID_CONFIG_PATH  :', HYBRID_CONFIG_PATH)


In [ ]:
# ── Cell 8: Train baseline ─────────────────────────────────────────────────
train_env = build_runtime_env()
train_env.update({'OMP_NUM_THREADS': str(TRAIN_NUM_WORKERS), 'MKL_NUM_THREADS': str(TRAIN_NUM_WORKERS)})
baseline_cmd = [
    'uv', 'run', '--no-sync', '--python', str(VENV_PY),
    'accelerate', 'launch',
    f'--num_processes={ACCELERATE_NUM_PROCESSES}',
    f'--mixed_precision={ACCELERATE_MIXED_PRECISION}',
    '--dynamo_backend=no',
    'src/f5_tts/train/train.py',
    '--config-name', BASELINE_CONFIG_NAME,
]
run_cmd(baseline_cmd, cwd=REPO_DIR, env=train_env)
run_cmd(['ls', '-lah', str(BASELINE_SAVE_DIR)])


In [ ]:
# ── Cell 9: Train hybrid flash-attn ───────────────────────────────────────
train_env = build_runtime_env()
train_env.update({'OMP_NUM_THREADS': str(TRAIN_NUM_WORKERS), 'MKL_NUM_THREADS': str(TRAIN_NUM_WORKERS)})
run_py_nosync(['-c', "import flash_attn; print('flash_attn ok', flash_attn.__version__)"], cwd=REPO_DIR)
hybrid_cmd = [
    'uv', 'run', '--no-sync', '--python', str(VENV_PY),
    'accelerate', 'launch',
    f'--num_processes={ACCELERATE_NUM_PROCESSES}',
    f'--mixed_precision={ACCELERATE_MIXED_PRECISION}',
    '--dynamo_backend=no',
    'src/f5_tts/train/train.py',
    '--config-name', HYBRID_CONFIG_NAME,
]
run_cmd(hybrid_cmd, cwd=REPO_DIR, env=train_env)
run_cmd(['ls', '-lah', str(HYBRID_SAVE_DIR)])


In [ ]:
# ── Cell 10: Benchmark baseline vs hybrid ─────────────────────────────────
bench = run_py_nosync([
    'src/f5_tts/scripts/benchmark_hybrid_mamba.py',
    '--device', 'cuda',
    '--batch-size', str(BENCHMARK_BATCH_SIZE),
    '--frame-length', str(BENCHMARK_FRAME_LENGTH),
    '--long-frame-length', str(BENCHMARK_LONG_FRAME_LENGTH),
    '--text-length', str(BENCHMARK_TEXT_LENGTH),
    '--sample-steps', str(BENCHMARK_SAMPLE_STEPS),
    '--warmup-iters', str(BENCHMARK_WARMUP_ITERS),
    '--iters', str(BENCHMARK_ITERS),
    '--baseline-attn-backend', 'torch',
    '--hybrid-attn-backend', 'flash_attn',
], cwd=REPO_DIR, capture_output=True)
print(bench.stdout)
BENCHMARK_JSON.write_text(bench.stdout, encoding='utf-8')
print('Saved benchmark JSON:', BENCHMARK_JSON)


In [ ]:
# ── Cell 11: Inference + objective metrics + subjective pack ──────────────
def infer_one(model_label: str, config_name: str, ckpt_path: Path, spec: dict) -> Path:
    out_dir = GENERATED_ROOT / model_label / spec['name']
    out_dir.mkdir(parents=True, exist_ok=True)
    cmd = [
        'uv', 'run', '--no-sync', '--python', str(VENV_PY),
        'accelerate', 'launch',
        f'--num_processes={ACCELERATE_NUM_PROCESSES}',
        f'--mixed_precision={ACCELERATE_MIXED_PRECISION}',
        '--dynamo_backend=no',
        'src/f5_tts/eval/eval_infer_batch.py',
        '--config_path', str(REPO_DIR / 'src/f5_tts/configs' / config_name),
        '--ckpt_path', str(ckpt_path),
        '--output_dir', str(out_dir),
        '--meta_file', spec['meta_file'],
        '-s', str(INFER_SEED),
        '-n', config_name.rsplit('.', 1)[0],
        '-c', '0',
        '-nfe', str(INFER_NFE_STEP),
        '-t', spec['name'],
    ]
    if spec['task_type'] == 'librispeech':
        cmd.extend(['-p', str(spec['librispeech_test_clean_path'])])
    run_cmd(cmd, cwd=REPO_DIR, env=build_runtime_env())
    return out_dir


def run_objective_eval(spec: dict, gen_dir: Path):
    if spec['task_type'] == 'seedtts':
        base_cmd = ['src/f5_tts/eval/eval_seedtts_testset.py', '--meta_file', spec['meta_file'], '--wavlm_ckpt_dir', str(WAVLM_CKPT), '-l', spec['lang'], '-g', str(gen_dir), '-n', EVAL_GPU_LIST]
        if ASR_CKPT is not None:
            base_cmd.extend(['--asr_ckpt_dir', str(ASR_CKPT)])
        run_py_nosync(base_cmd + ['-e', 'wer'], cwd=REPO_DIR)
        run_py_nosync(base_cmd + ['-e', 'sim'], cwd=REPO_DIR)
    else:
        base_cmd = ['src/f5_tts/eval/eval_librispeech_test_clean.py', '--meta_file', spec['meta_file'], '--wavlm_ckpt_dir', str(WAVLM_CKPT), '-g', str(gen_dir), '-n', EVAL_GPU_LIST, '-p', str(spec['librispeech_test_clean_path'])]
        if ASR_CKPT is not None:
            base_cmd.extend(['--asr_ckpt_dir', str(ASR_CKPT)])
        run_py_nosync(base_cmd + ['-e', 'wer'], cwd=REPO_DIR)
        run_py_nosync(base_cmd + ['-e', 'sim'], cwd=REPO_DIR)


baseline_ckpt = resolve_latest_checkpoint(BASELINE_SAVE_DIR)
hybrid_ckpt = resolve_latest_checkpoint(HYBRID_SAVE_DIR)

for spec in EVAL_SPECS:
    base_dir = infer_one('baseline', BASELINE_CONFIG_NAME, baseline_ckpt, spec)
    hybrid_dir = infer_one('hybrid', HYBRID_CONFIG_NAME, hybrid_ckpt, spec)
    run_objective_eval(spec, base_dir)
    run_objective_eval(spec, hybrid_dir)

    subjective_dir = SUBJECTIVE_ROOT / spec['name']
    prep_cmd = ['src/f5_tts/scripts/prepare_subjective_eval.py', '--baseline-dir', str(base_dir), '--hybrid-dir', str(hybrid_dir), '--task', spec['task_type'], '--meta-file', spec['meta_file'], '--output-dir', str(subjective_dir), '--baseline-name', 'baseline_f5', '--hybrid-name', 'hybrid_f5_mamba']
    if spec['task_type'] == 'librispeech':
        prep_cmd.extend(['--librispeech-test-clean-path', str(spec['librispeech_test_clean_path'])])
    run_py_nosync(prep_cmd, cwd=REPO_DIR)

    matrix_dir = MATRIX_ROOT / spec['name']
    build_cmd = ['src/f5_tts/scripts/build_eval_matrix.py', '--baseline-dir', str(base_dir), '--hybrid-dir', str(hybrid_dir), '--output-dir', str(matrix_dir), '--baseline-name', 'baseline_f5', '--hybrid-name', 'hybrid_f5_mamba']
    cmos_key = subjective_dir / 'cmos_key.csv'
    if SMOS_RATINGS is not None and SMOS_RATINGS.exists():
        build_cmd.extend(['--smos-ratings', str(SMOS_RATINGS)])
    if CMOS_RATINGS is not None and CMOS_RATINGS.exists() and cmos_key.exists():
        build_cmd.extend(['--cmos-ratings', str(CMOS_RATINGS), '--cmos-key', str(cmos_key)])
    run_py_nosync(build_cmd, cwd=REPO_DIR)


In [ ]:
# ── Cell 12: Ringkasan hasil ───────────────────────────────────────────────
all_rows = []
for spec in EVAL_SPECS:
    matrix_json = MATRIX_ROOT / spec['name'] / 'evaluation_matrix.json'
    if matrix_json.exists():
        rows = json.loads(matrix_json.read_text(encoding='utf-8'))
        for row in rows:
            item = dict(row)
            item['task'] = spec['name']
            all_rows.append(item)

print(json.dumps(all_rows, indent=2, ensure_ascii=False))
print('Matrix root    :', MATRIX_ROOT)
print('Subjective root:', SUBJECTIVE_ROOT)
print('Benchmark json :', BENCHMARK_JSON)
print('Isi smos_manifest.csv dan cmos_blind_manifest.csv kalau mau skor manusia masuk ke matrix.')


## Notes

- Notebook ini dibuat sesingkat mungkin supaya tetap terasa seperti `traineo kaggle clean`.
- Baseline tetap `torch`; hybrid tetap `flash_attn`.
- `WER` dan `SIM-o` otomatis. `SMOS` dan `CMOS` tetap butuh rating manusia.